# Lecture 8: Multivariate Time Series
In this notebook we will cover:
- Cross-covariance matrix and CCF
- VAR(p) model: estimation and interpretation
- Lag length selection
- Granger causality test
- Cointegration (Engle-Granger)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf, plot_ccf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
print("Ready.")


## 8.1 Properties of Multivariate Time Series

**Cross-covariance function (CCVF):**
$$\gamma_{XY}(h) = \text{Cov}(X_{t+h}, Y_t) = E[(X_{t+h}-\mu_X)(Y_t-\mu_Y)]$$

**Cross-correlation function (CCF):**
$$\rho_{XY}(h) = \frac{\gamma_{XY}(h)}{\sqrt{\gamma_X(0)\gamma_Y(0)}}$$

**Important:** $\rho_{XY}(h) \neq \rho_{YX}(h)$ — the lag direction matters!
- $\rho_{XY}(h) \neq 0$, $h > 0$: $X$ is leading $Y$ with a lag


In [ ]:
# ── CCF and Lead-Lag Relationship ──

np.random.seed(42)
n = 300

# X is a leading indicator, Y follows with a 3-period lag
eps_x = np.random.normal(0, 1, n)
eps_y = np.random.normal(0, 0.5, n)
X = np.zeros(n)
Y = np.zeros(n)
for t in range(1, n):
    X[t] = 0.7*X[t-1] + eps_x[t]

for t in range(4, n):
    Y[t] = 0.5*X[t-3] + 0.4*Y[t-1] + eps_y[t]  # Y lags X by 3 periods

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].plot(X, label='X (Leading)', color='steelblue', lw=0.8)
axes[0,0].plot(Y, label='Y (Lagging)', color='darkorange', lw=0.8)
axes[0,0].set_title('X and Y Time Series', fontweight='bold')
axes[0,0].legend()

# CCF
lags = np.arange(-20, 21)
cc = [np.corrcoef(X[max(0,-h):n-max(0,h)], Y[max(0,h):n-max(0,-h)])[0,1] for h in lags]
axes[0,1].bar(lags, cc, color='steelblue', alpha=0.7, width=0.8)
axes[0,1].axhline(0, color='black', lw=0.5)
conf = 1.96/np.sqrt(n)
axes[0,1].axhline(conf, color='red', ls='--', alpha=0.7, label='95% CI')
axes[0,1].axhline(-conf, color='red', ls='--', alpha=0.7)
axes[0,1].set_title('CCF: cross-correlation between X and Y', fontweight='bold')
axes[0,1].set_xlabel('Lag h')
axes[0,1].axvline(-3, color='green', ls=':', lw=2, label='h=-3 (expected)')
axes[0,1].legend(fontsize=8)

# ACF for both series
plot_acf(X, lags=20, ax=axes[1,0], title='ACF — X')
plot_acf(Y, lags=20, ax=axes[1,1], title='ACF — Y')

plt.tight_layout()
plt.show()
print("Pronounced CCF around h=-3: Y lags X by 3 periods.")


## 8.2 VAR(p) Model

**Vector Autoregressive Model VAR(p):**

$$\mathbf{X}_t = \mathbf{c} + \Phi_1 \mathbf{X}_{t-1} + \Phi_2 \mathbf{X}_{t-2} + \cdots + \Phi_p \mathbf{X}_{t-p} + \mathbf{Z}_t$$

where:
- $\mathbf{X}_t = (X_{1t}, X_{2t}, \ldots, X_{kt})^T$: $k$-dimensional process
- $\Phi_i$: $k \times k$ coefficient matrices
- $\mathbf{Z}_t \sim WN(\mathbf{0}, \Sigma)$: multivariate white noise

**Lag selection:** AIC, BIC, FPE, HQIC criteria.


In [ ]:
# ── VAR(2) Simulation and Estimation ──

np.random.seed(42)
n = 400

# VAR(2) process: 2 variables, 2 lags
Phi1 = np.array([[0.5, 0.2],
                  [0.1, 0.6]])
Phi2 = np.array([[-0.1, 0.05],
                  [0.05, -0.15]])
Sigma = np.array([[1.0, 0.5],
                   [0.5, 1.0]])

X = np.zeros((n, 2))
for t in range(2, n):
    noise = np.random.multivariate_normal([0, 0], Sigma)
    X[t] = Phi1 @ X[t-1] + Phi2 @ X[t-2] + noise

df = pd.DataFrame(X, columns=['X1', 'X2'])

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(df['X1'], color='steelblue', lw=0.8, label='X1')
axes[1].plot(df['X2'], color='darkorange', lw=0.8, label='X2')
axes[0].set_title('VAR(2) Simulated Data — X1', fontweight='bold')
axes[1].set_title('VAR(2) Simulated Data — X2', fontweight='bold')
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

# Model estimation
model = VAR(df)

# Lag selection
lag_order = model.select_order(maxlags=8)
print("Lag Selection Criteria:")
print(lag_order.summary())


In [ ]:
# ── VAR(2) Coefficient Estimation ──

var_fit = model.fit(maxlags=2, ic=None)
print(var_fit.summary())


In [ ]:
# ── VAR Forecasting ──

forecast_steps = 20
forecast = var_fit.forecast(df.values[-var_fit.k_ar:], steps=forecast_steps)
forecast_df = pd.DataFrame(forecast, columns=['X1_pred', 'X2_pred'])

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

for i, (col, pred_col, color) in enumerate([('X1','X1_pred','steelblue'),
                                              ('X2','X2_pred','darkorange')]):
    axes[i].plot(range(n), df[col], color=color, lw=0.8, alpha=0.6, label='Observations')
    pred_idx = range(n, n+forecast_steps)
    axes[i].plot(pred_idx, forecast_df[pred_col], color='red', lw=2, label='Forecast')
    axes[i].axvline(n, color='black', ls='--', alpha=0.5)
    axes[i].set_title(f'{col} — VAR(2) {forecast_steps}-Step Forecast', fontweight='bold')
    axes[i].legend()

plt.tight_layout()
plt.show()


## 8.3 Granger Causality Test

**Granger Causality:** $X$ Granger-causes $Y$ if the past values of $X$ help forecast $Y$, beyond what $Y$'s own past provides.

$$H_0: X\text{ does not Granger-cause }Y$$

**Note:** This measures predictive causality, not physical causation!


In [ ]:
# ── Granger Causality Test ──

print("Granger Causality: X1 -> X2 ?")
print("="*50)
gc_result = grangercausalitytests(df[['X2','X1']], maxlag=4, verbose=True)


## 8.4 Cointegration

If two $I(1)$ (unit-root) series $X_t$ and $Y_t$ are individually non-stationary but a linear combination $Y_t - \alpha X_t$ is stationary, the series are **cointegrated**.

**Economic examples:** Prices and costs, interest rates, exchange rate parity.

**Engle-Granger Test:**
1. Regress $Y_t = \alpha + \beta X_t + u_t$
2. Apply ADF test to residuals $\hat{u}_t$


In [ ]:
# ── Cointegration Example ──

np.random.seed(123)
n = 300

# Common stochastic trend (cointegrated pair)
common_trend = np.cumsum(np.random.normal(0, 1, n))
X_coint = common_trend + np.random.normal(0, 0.5, n)
Y_coint = 2 * common_trend + 1 + np.random.normal(0, 0.5, n)

# Independent I(1) series (not cointegrated)
X_indep = np.cumsum(np.random.normal(0, 1, n))
Y_indep = np.cumsum(np.random.normal(0, 1, n))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0,0].plot(X_coint, label='X', color='steelblue')
axes[0,0].plot(Y_coint, label='Y', color='darkorange')
axes[0,0].set_title('Cointegrated Pair', fontweight='bold')
axes[0,0].legend()

# Residuals
resid_coint = Y_coint - 2*X_coint - 1
axes[0,1].plot(resid_coint, color='green')
axes[0,1].axhline(0, color='red', ls='--')
axes[0,1].set_title('Residuals Y - 2X - 1 (Should be Stationary)', fontweight='bold')

axes[1,0].plot(X_indep, label='X', color='steelblue')
axes[1,0].plot(Y_indep, label='Y', color='darkorange')
axes[1,0].set_title('Independent I(1) Series (Spurious Regression)', fontweight='bold')
axes[1,0].legend()

resid_indep = Y_indep - X_indep
axes[1,1].plot(resid_indep, color='red')
axes[1,1].axhline(0, color='black', ls='--')
axes[1,1].set_title('Residuals Y - X (Not Stationary)', fontweight='bold')

plt.tight_layout()
plt.show()

# Engle-Granger test
t1, p1, cv1 = coint(X_coint, Y_coint)
t2, p2, cv2 = coint(X_indep, Y_indep)
print(f"Cointegrated pair  — p-value: {p1:.4f} {'=> Cointegrated' if p1<0.05 else '=> Not cointegrated'}")
print(f"Independent pair   — p-value: {p2:.4f} {'=> Cointegrated' if p2<0.05 else '=> Not cointegrated'}")


## Summary

| Method | Use |
|--------|-----|
| **CCF** | Lead-lag relationship between series |
| **VAR(p)** | Multivariate forecasting |
| **Granger** | Predictive causality test |
| **Cointegration** | Long-run equilibrium relationship |

**Model Selection:**
- Lag order: AIC/BIC/HQIC
- Stationarity check: ADF test
- If cointegrated: use VECM instead of VAR
